# 04 — Direct integration versus IMU preintegration

**Learning objective:** integrate the same calibrated specific force
and angular velocity in two ways. Direct integration updates pose,
velocity and position each sample. Preintegration sums motion relative
to each interval's **starting sensor frame**; gravity is added only when
combining an interval with a world-frame state.

Both methods drift: preintegration is a convenient *measurement factor*,
not a gravity/bias correction or a complete sensor-fusion solution.
Input CSV must contain calibrated acceleration in m/s² and gyro in rad/s
in the **same right-handed body axes**. Set `DATA_FILE` to your real run.
If an overflow was detected, loading fails rather than hiding a gap.

In [ ]:
try:
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.spatial.transform import Rotation
    from imu_lab import (load_csv, estimate_initial_state,
                         integrate_direct, integrate_preintegrated)
    from imu_lab.recording import verify_capture_metadata
    DATA_FILE = Path('data/synthetic_noisy.csv')
    if not DATA_FILE.exists():
        from create_synthetic_data import main as generate_example_data
        generate_example_data()  # local, deterministic demo only
    # Keep the sensor completely STILL for at least the first 2 s.
    verify_capture_metadata(DATA_FILE)
    imu = load_csv(DATA_FILE)
    R0, remaining_gyro_bias = estimate_initial_state(imu, static_seconds=2.0)
    print('Samples:', len(imu.t), 'duration:', imu.t[-1] - imu.t[0], 's')
    print('Residual gyro bias [rad/s]:', remaining_gyro_bias)
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

In [ ]:
try:
    # Keyframe stride = 20 samples; at 200 Hz this would be 0.1 s.
    # Synthetic demo runs at 100 Hz, so here it is 0.2 s.
    STRIDE = 20
    direct = integrate_direct(imu, R0=R0, gyro_bias=remaining_gyro_bias)
    pre, blocks = integrate_preintegrated(
        imu, every_n_samples=STRIDE, R0=R0, gyro_bias=remaining_gyro_bias)
    inds = [0] + [b.end_idx for b in blocks]
    print('Direct / preintegration max mismatch at keyframes [m]:',
          np.max(np.linalg.norm(pre.p - direct.p[inds], axis=1)))
    if blocks:
        first = blocks[0]
        print('First block duration [s]:', first.duration_s)
        print('delta-v [start body m/s]:', first.delta_v)
        print('delta-p [start body m]:', first.delta_p)
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

In [ ]:
try:
    fig = plt.figure(figsize=(12, 5))
    ax = fig.add_subplot(121)
    ax.plot(direct.p[:,0], direct.p[:,1], label='Direct, every IMU sample')
    ax.plot(pre.p[:,0], pre.p[:,1], 'o', ms=2, label='Preintegrated keyframes')
    ax.set(xlabel='X [m]', ylabel='Y [m]', title='XY estimated trajectory')
    ax.axis('equal'); ax.legend()
    ax3 = fig.add_subplot(122, projection='3d')
    ax3.plot(*direct.p.T, label='Direct')
    ax3.plot(*pre.p.T, 'o', ms=2, label='Keyframes')
    ax3.set(xlabel='X [m]', ylabel='Y [m]', zlabel='Z [m]', title='3D path')
    fig.tight_layout(); plt.show()
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

### Suggested experiments

1. Change `STRIDE` to 1, 10, 50, 100 and compare endpoint agreement.
2. Switch to `data/synthetic_ideal.csv` (same known square path).
3. Record **20 seconds still** and inspect drift before trying movement.
4. Move the sensor about 30 cm without rotating and return to rest.
5. Rotate in place: ideally translation remains zero; in practice attitude
   and axis misalignment cause false horizontal acceleration.

With no magnetometer or external reference, yaw is not observable from a
stationary accelerometer. Initial yaw is assigned arbitrarily to zero.
See `imu_lab/core.py` for equations and frame conventions.